In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import random
import ale_py
import wandb

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import seaborn as sns
import collections

sns.set()
sns.set_context("notebook")
sns.set_style("whitegrid")
gym.register_envs(ale_py)


env = gym.make("ALE/Asterix-v5", obs_type="grayscale", frameskip=1)
env = gym.wrappers.AtariPreprocessing(env, frame_skip=4)
env = gym.wrappers.FrameStackObservation(env, 4)

obs, info = env.reset()



for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

env.close()



class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)
    

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

replay_buffer = collections.deque(maxlen=100000)


    

In [12]:
wandb.login("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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\acrab\_netrc


True

In [13]:
config = {
    "learning rate": 1e-4,
    "gamma": 0.99,
    "episodes": 10_000,
}

In [14]:
def process_obs(obs):
    obs = np.array(obs, dtype=np.float32) / 255.0 
    return torch.tensor(obs, device=device).unsqueeze(0)

In [ ]:
#Empty start

episode_rewards = []
train_losses = []
step_count = 0

n_actions = env.action_space.n
Q_net = DQN(n_actions).to(device)
T_net = DQN(n_actions).to(device)
T_net.load_state_dict(Q_net.state_dict())
T_net.eval()

optimizer = optim.Adam(Q_net.parameters(), lr=config['learning rate'])


action_space = env.action_space


print("Filling buffer")
while len(replay_buffer) < 5000:
    obs, info = env.reset()
    obs = process_obs(obs)
    done = False
    while not done and len(replay_buffer) < 5000:
        action = action_space.sample()
        next_obs, reward, terminated, truncated, info = env.step(action)
        reward = reward / 50.0
        next_obs = process_obs(next_obs)
        done = terminated or truncated
        replay_buffer.append([obs.cpu().numpy(), action, reward, next_obs.cpu().numpy(), terminated, truncated])
        obs = next_obs
print(f"Buffer filled")

start_episode = 0

Warming up replay buffer...
Warmup done — buffer size: 5000


In [ ]:



def train_step(obss, actions, y_vals):
    q_vals = Q_net(obss)
    chosen_q = q_vals.gather(1, actions.unsqueeze(1)).squeeze(1)
    td_error = torch.clamp(y_vals - chosen_q, -1, 1)
    loss = (td_error ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()



def save_checkpoint(path, Q_net, T_net, optimizer, step_count, episode_rewards, train_losses, episode):
    torch.save({
        'episode': episode,
        'step_count': step_count,
        'Q_net': Q_net.state_dict(),
        'T_net': T_net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'episode_rewards': episode_rewards,
        'train_losses': train_losses,
    }, path)
    print(f"Checkpoint saved — episode {episode}, steps {step_count}")

def load_checkpoint(path, Q_net, T_net, optimizer):
    checkpoint = torch.load(path)
    Q_net.load_state_dict(checkpoint['Q_net'])
    T_net.load_state_dict(checkpoint['T_net'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    return (
        checkpoint['step_count'],
        checkpoint['episode_rewards'],
        checkpoint['train_losses'],
        checkpoint['episode']
    )

For Loading a model start here

In [ ]:
#load start

step_count, episode_rewards, train_losses, start_episode = load_checkpoint(
    "ddqn_checkpoint_ep5000.pt", Q_net, T_net, optimizer
)

In [ ]:
print("Starting refill")
while len(replay_buffer) < 100000:
    obs, info = env.reset()
    obs = process_obs(obs)
    done = False
    while not done:
        with torch.no_grad():
            action = Q_net(obs).argmax(dim=1).item()
        next_obs, reward, terminated, truncated, info = env.step(action)
        reward = reward / 50.0
        next_obs = process_obs(next_obs)
        done = terminated or truncated
        replay_buffer.append([obs.cpu().numpy(), action, reward,
                               next_obs.cpu().numpy(), terminated, truncated])
        obs = next_obs
print(f"Buffer refilled")


Refilling buffer with on-policy experience...
Buffer refilled: 50025 transitions


In [ ]:
replay_buffer = collections.deque(maxlen=100000)
discount = config['gamma']
action_space = env.action_space


wandb.init(
            project="RL - Final Assignment",
            #entity="a-crabtree",          
            sync_tensorboard=True,
            config=config,
            name="DDQN",
            monitor_gym=True,
            save_code=True,
        )


x = start_episode

for x in range(start_episode, start_episode + config['episodes']):
    #eps = max(0.1, 1.0 - step_count / 100000)
    eps = max(0.05, 1.0 - step_count / 500000)

    obs, info = env.reset()
    obs = process_obs(obs)
    done = False
    total_reward = 0
    total_reward_unscaled = 0
        

    if x % 20 == 0 and x > 0:
        recent_mean = np.mean(episode_rewards[-20:])
        recent_best = np.max(episode_rewards[-20:])
        print(f"Episode {x:4d} | eps={eps:.3f} | "
              f"mean={recent_mean:.1f} | best={recent_best:.1f} | "
              f"steps={step_count}")

    while not done:
        step_count += 1

        if step_count % 5000 == 0:
            T_net.load_state_dict(Q_net.state_dict())

        if random.random() > eps:
            with torch.no_grad():
                action = Q_net(obs).argmax(dim=1).item()
        else:
            action = action_space.sample()

        next_obs, reward, terminated, truncated, info = env.step(action)
        reward = reward / 50.0
        next_obs = process_obs(next_obs)
        done = terminated or truncated

        replay_buffer.append([obs.cpu().numpy(), action, reward, next_obs.cpu().numpy(), terminated, truncated])
        total_reward += reward
        total_reward_unscaled += reward * 50
        obs = next_obs

        if len(replay_buffer) > 1000:
            replays = random.sample(replay_buffer, 64)

            obss = torch.tensor(np.vstack([r[0] for r in replays]), device=device)
            actions = torch.tensor([r[1] for r in replays], device=device, dtype=torch.long)
            rewards = torch.tensor([r[2] for r in replays], device=device, dtype=torch.float32)
            next_obss = torch.tensor(np.vstack([r[3] for r in replays]), device=device)
            dones = torch.tensor([r[4] or r[5] for r in replays], device=device, dtype=torch.float32)

            with torch.no_grad():
                next_actions = Q_net(next_obss).argmax(dim=1)
                q_vals = T_net(next_obss).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            y_vals = rewards + discount * q_vals * (1 - dones)

            loss = train_step(obss, actions, y_vals)    
            train_losses.append(loss)

    episode_rewards.append(total_reward_unscaled)
    print(f"Episode {x}, Reward: {total_reward_unscaled:.0f}")

    mean_loss = np.mean(train_losses[-100:]) if len(train_losses) > 0 else 0

    wandb.log({
    "episode": x,
    "reward": total_reward_unscaled,
    "epsilon": eps,
    "mean_loss": mean_loss,
    "buffer_size": len(replay_buffer),
    "step_count": step_count
})

    if x % 500 == 0 and x > 0:
        save_checkpoint(f"ddqn_checkpoint_ep{x}.pt", 
                    Q_net, T_net, optimizer, 
                    step_count, episode_rewards, train_losses, x)
        print(f"Checkpoint saved at episode {x}")

save_checkpoint(f"ddqn_checkpoint_ep{x}.pt", 
                    Q_net, T_net, optimizer, 
                    step_count, episode_rewards, train_losses, x)
print(f"Checkpoint saved at episode {x}")
env.close()

wandb.finish()

Episode 0, Reward: 250
Episode 1, Reward: 350
Episode 2, Reward: 150
Episode 3, Reward: 450
Episode 4, Reward: 100
Episode 5, Reward: 150
Episode 6, Reward: 300
Episode 7, Reward: 400
Episode 8, Reward: 100
Episode 9, Reward: 250
Episode 10, Reward: 350
Episode 11, Reward: 100
Episode 12, Reward: 300
Episode 13, Reward: 500
Episode 14, Reward: 150
Episode 15, Reward: 150
Episode 16, Reward: 250
Episode 17, Reward: 300
Episode 18, Reward: 600
Episode 19, Reward: 250
Episode   20 | eps=0.990 | mean=272.5 | best=600.0 | steps=5073
Episode 20, Reward: 150
Episode 21, Reward: 400
Episode 22, Reward: 100
Episode 23, Reward: 250
Episode 24, Reward: 200
Episode 25, Reward: 350
Episode 26, Reward: 250
Episode 27, Reward: 150
Episode 28, Reward: 150
Episode 29, Reward: 350
Episode 30, Reward: 400
Episode 31, Reward: 250
Episode 32, Reward: 400
Episode 33, Reward: 350
Episode 34, Reward: 400
Episode 35, Reward: 300
Episode 36, Reward: 300
Episode 37, Reward: 200
Episode 38, Reward: 400
Episode 39

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Checkpoint saved — episode 9999, steps 3052879
Checkpoint saved at episode 9999


wandb: WARNING Artifact "source-RL_-_Final_Assignment-c__Users_acrab_OneDrive_Documents_GitHub_DQN_DDQN.ipynb" already exists with the same content. No new version will be created.


buffer_size,▁▃▆█████████████████████████████████████
episode,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███████
epsilon,█▆▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
mean_loss,▆▂▅▆▅▄▁▂▄▂▇▃▃▄▂▄▁▃▂▂▇▁█▅▇▅▄▆▃▄▆▆▆▆▅▃▂▅▄▅
reward,▁▁▁▁▁▂▃▅▂▆▃▅▄▇▅▇▅▄▇▆▂▂▂▃▅▆▃▅▅▅▄▅▃▆█▄▅▃▇▅
step_count,▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇███
buffer_size,100000
episode,9999
epsilon,0.05
mean_loss,0.04436
reward,800
